In [ ]:
pip install whisperx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 

In [ ]:
!pip install ffmpeg

  Preparing metadata (setup.py) ... done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6126 sha256=7cc8c2ef18b0474e875a5864fd6ebb1cbe0a77284bb326b4489435f78d73e018
  Stored in directory: /root/.cache/pip/wheels/a4/04/6c/ab972358c48aedc5be02f1d28968f2baa491a2837270932043
Successfully built ffmpeg


In [ ]:
pip install setuptools-rust

In [ ]:
import whisperx
import gc
import torch

# Set parameters
device = "cuda" if torch.cuda.is_available() else "cpu"
audio_file = "YOUR_AUDIO_FILE.mp3"  # Replace with your audio file path
batch_size = 16  # reduce if low on GPU mem
compute_type = "float16" if device == "cuda" else "int8"  # change to "int8" if low on GPU mem

# 1. Transcribe with original whisper (batched)
model = whisperx.load_model("large-v2", device, compute_type=compute_type)
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)
print("--- Transcription (Before Alignment) ---")
print(result["segments"])

# Free GPU resources
gc.collect(); torch.cuda.empty_cache(); del model

# 2. Align whisper output
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
print("\n--- Transcription (After Alignment) ---")
print(result["segments"])

# Free GPU resources
gc.collect(); torch.cuda.empty_cache(); del model_a

# 3. Assign speaker labels (Diarization)
# Note: You need a Hugging Face token for pyannote.audio
YOUR_HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN" # Replace with your token
diarize_model = whisperx.DiarizationPipeline(use_auth_token=YOUR_HF_TOKEN, device=device)

diarize_segments = diarize_model(audio)
result = whisperx.assign_word_speakers(diarize_segments, result)
print("\n--- Final Result with Speaker IDs ---")
print(result["segments"])
